In [0]:
from pyspark.sql import functions as F

# Lê as tabelas brutas persistidas na etapa anterior
partidas_brutas = spark.table(
    "workspace.mvp_gramados.bronze_partidas"
)

gramados_brutos = spark.table(
    "workspace.mvp_gramados.bronze_gramados"
)

# Mostra os nomes das colunas e seus tipos
partidas_brutas.printSchema()
gramados_brutos.printSchema()

In [0]:
partidas_tratadas = (
    partidas_brutas
    .withColumnRenamed("rodata", "rodada")
    .withColumn("ID", F.expr("try_cast(ID AS INT)"))
    .withColumn("rodada", F.expr("try_cast(rodada AS INT)"))
    .withColumn(
        "data",
        F.expr("cast(try_to_timestamp(data, 'dd/MM/yyyy') AS DATE)")
    )
    .withColumn(
        "mandante_Placar",
        F.expr("try_cast(mandante_Placar AS INT)")
    )
    .withColumn(
        "visitante_Placar",
        F.expr("try_cast(visitante_Placar AS INT)")
    )
)

# Confere se alguma data ficou ausente após a conversão
display(
    partidas_tratadas.agg(
        F.count("*").alias("total_partidas"),
        F.sum(
            F.when(F.col("data").isNull(), 1).otherwise(0)
        ).alias("datas_ausentes_ou_invalidas")
    )
)

# Mostra uma amostra com os campos convertidos
display(
    partidas_tratadas.select(
        "ID", "rodada", "data", "mandante", "visitante",
        "mandante_Placar", "visitante_Placar"
    ).orderBy("ID").limit(5)
)

In [0]:
# Seleciona os dois anos da análise
partidas_2023_2024 = partidas_tratadas.filter(
    F.col("data").between("2023-01-01", "2024-12-31")
)

# Identifica placares ausentes ou negativos
placar_invalido = (
    F.col("mandante_Placar").isNull()
    | F.col("visitante_Placar").isNull()
    | (F.col("mandante_Placar") < 0)
    | (F.col("visitante_Placar") < 0)
)

display(
    partidas_2023_2024.agg(
        F.count("*").alias("total_partidas"),
        F.countDistinct("ID").alias("ids_distintos"),
        F.sum(
            F.when(F.col("ID").isNull(), 1).otherwise(0)
        ).alias("ids_ausentes"),
        F.sum(
            F.when(placar_invalido, 1).otherwise(0)
        ).alias("placares_ausentes_ou_invalidos")
    )
)

# Confere a quantidade por ano
display(
    partidas_2023_2024
    .groupBy(F.year("data").alias("ano"))
    .count()
    .orderBy("ano")
)

In [0]:
# Seleciona os dois anos da análise
partidas_2023_2024 = partidas_tratadas.filter(
    F.col("data").between("2023-01-01", "2024-12-31")
)

# Identifica placares ausentes ou negativos
placar_invalido = (
    F.col("mandante_Placar").isNull()
    | F.col("visitante_Placar").isNull()
    | (F.col("mandante_Placar") < 0)
    | (F.col("visitante_Placar") < 0)
)

display(
    partidas_2023_2024.agg(
        F.count("*").alias("total_partidas"),
        F.countDistinct("ID").alias("ids_distintos"),
        F.sum(
            F.when(F.col("ID").isNull(), 1).otherwise(0)
        ).alias("ids_ausentes"),
        F.sum(
            F.when(placar_invalido, 1).otherwise(0)
        ).alias("placares_ausentes_ou_invalidos")
    )
)

# Confere a quantidade por ano
display(
    partidas_2023_2024
    .groupBy(F.year("data").alias("ano"))
    .count()
    .orderBy("ano")
)

In [0]:
gramados_tratados = (
    gramados_brutos
    .withColumn(
        "inicio_validade",
        F.expr("try_cast(inicio_validade AS DATE)")
    )
    .withColumn(
        "fim_validade",
        F.expr("try_cast(fim_validade AS DATE)")
    )
)

# Uma classificação precisa de datas preenchidas e em ordem
periodo_problematico = (
    F.col("inicio_validade").isNull()
    | F.col("fim_validade").isNull()
    | (F.col("inicio_validade") > F.col("fim_validade"))
)

display(
    gramados_tratados.agg(
        F.count("*").alias("total_registros"),
        F.countDistinct("arena").alias("arenas_distintas"),
        F.sum(
            F.when(
                F.col("tipo_gramado") == "nao_confirmado", 1
            ).otherwise(0)
        ).alias("registros_pendentes"),
        F.sum(
            F.when(
                (F.col("tipo_gramado") != "nao_confirmado")
                & periodo_problematico,
                1
            ).otherwise(0)
        ).alias("classificados_com_periodo_problematico")
    )
)

# Exibe os registros cuja pesquisa ainda está pendente
display(
    gramados_tratados
    .filter(F.col("tipo_gramado") == "nao_confirmado")
    .select("arena", "tipo_gramado", "inicio_validade", "fim_validade")
)

In [0]:
# Cruza pelo nome exato do estádio.
# "left" mantém todas as 760 partidas.
partidas_gramados = (
    partidas_2023_2024
    .join(gramados_tratados, on="arena", how="left")
    .withColumn(
        "status_gramado",
        F.when(
            F.col("tipo_gramado").isNull(),
            "sem_classificacao"
        )
        .when(
            F.col("tipo_gramado") == "nao_confirmado",
            "pesquisa_pendente"
        )
        .when(
            F.col("inicio_validade").isNull()
            | F.col("fim_validade").isNull(),
            "periodo_nao_informado"
        )
        .when(
            (F.col("data") >= F.col("inicio_validade"))
            & (F.col("data") <= F.col("fim_validade")),
            "dentro_do_periodo"
        )
        .otherwise("fora_do_periodo")
    )
)

# Confere se o cruzamento multiplicou alguma partida
display(
    partidas_gramados.agg(
        F.count("*").alias("total_apos_cruzamento"),
        F.countDistinct("ID").alias("ids_distintos")
    )
)

# Mostra a cobertura da classificação
display(
    partidas_gramados
    .groupBy("status_gramado")
    .count()
    .orderBy("status_gramado")
)

In [0]:
# Salva a pesquisa com as datas convertidas
gramados_tratados.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_gramados.silver_gramados")

# Salva todas as partidas do recorte, incluindo as pendentes
partidas_gramados.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.mvp_gramados.silver_partidas_gramados")

# Confere o status diretamente na tabela salva
display(spark.sql("""
    SELECT
        status_gramado,
        COUNT(*) AS quantidade_partidas
    FROM workspace.mvp_gramados.silver_partidas_gramados
    GROUP BY status_gramado
    ORDER BY status_gramado
"""))